# Benchmark — Grounding DINO vs OmDet-Turbo sul PPE Tracker

Confronta i due detector open-vocabulary (Apache 2.0) **nel modo più fedele possibile all'uso reale**:
la pipeline completa che gira su un video. Tre sezioni, dalla più fedele alla più "da laboratorio":

| Sezione | Cosa misura | Fonte |
|---|---|---|
| **1. Tracker end-to-end** | throughput (FPS effettivo), stabilità identità (track creati), alert prodotti e loro composizione | `ppe_video.mp4`, pipeline completa |
| **2. Frame-level + weak ground truth** | accuratezza contro *fatti noti della scena* dichiarati a mano (quante persone, chi indossa cosa), flicker temporale, latenza | frame campionati dal video |
| **3. Ground truth vera (fallback)** | Precision / Recall / F1 per classe @ IoU 0.5 | test split etichettato `data/test` (4423 immagini, CC BY 4.0) |

> **Perché la sezione 2**: non esistono video PPE liberamente scaricabili con annotazioni per-frame.
> La "weak GT" è il compromesso più fedele all'uso: dichiari cosa è vero nel video (es. *"tutte le
> persone indossano casco e gilet"*) e misuri quanto spesso la pipeline è d'accordo.

**Tempi su CPU** (Intel Iris Xe): Grounding DINO ~22s/inferenza, OmDet-Turbo ~1.5s/inferenza.
Con i default il notebook completo richiede ~25 minuti (quasi tutti per Grounding DINO).

In [ ]:
import sys

sys.path.insert(0, "src")

import time
from collections import Counter, defaultdict
from pathlib import Path

import cv2 as cv
import matplotlib.pyplot as plt
import numpy as np
import yaml

from visual_security.analyzer import build_detector
from visual_security.person_ppe_checker import PersonPPEChecker
from visual_security.video_tracker import build_tracker

# ── Config ────────────────────────────────────────────────────────────────────
VIDEO = "ppe_video.mp4"
DETECTORS = ["grounding-dino", "omdet-turbo"]

# Sezione 1 - tracker end-to-end (stesso skip per un confronto alla pari;
# in produzione omdet-turbo puo' permettersi skip piu' basso)
SKIP_FRAMES = 8
PERSISTENCE, WINDOW = 3, 6
ZONES_FILE = None  # es. "zones.example.json"

# Sezione 2 - frame-level con weak ground truth.
# Dichiarare qui i FATTI NOTI della scena (guardare il video e adattare!):
WEAK_GT = {
    "persons_range": (1, 3),      # in ogni frame ci sono da 1 a 3 persone
    "helmet_worn_by_all": True,   # tutte le persone indossano il casco
    "vest_worn_by_all": True,     # tutte le persone indossano il gilet
}
VIDEO_SAMPLES = 10

# Sezione 3 - ground truth su immagini (campionamento stratificato per classe:
# il dataset ha annotazioni parziali, si valutano solo le classi annotate per immagine)
DATA_DIR = Path("data")
K_PER_CLASS = 6                   # immagini per classe canonica (Person/Helmet/Vest/Glove)
IOU_THR = 0.5
SEED = 42

assert Path(VIDEO).exists(), f"Video non trovato: {VIDEO}"
print("Config OK -", "dataset presente" if (DATA_DIR / "data.yaml").exists() else "dataset ASSENTE (sezione 3 saltata)")

## 1 · Tracker end-to-end sul video

La pipeline **completa** (detector → PPEChecker → PersonTracker → sliding window → alert) gira
sull'intero video, per ciascun detector, con gli stessi parametri. Metriche:

- **Wall time / FPS effettivo** — quanto costa processare il video per intero.
- **Track creati** — nel video ci sono sempre le stesse 2–3 persone: un buon detector+tracker crea
  pochi `track_id` (identità stabili); un detector instabile fa perdere i track e ne crea di nuovi.
- **Alert confermati** e composizione dei PPE mancanti — a parametri identici, più alert non
  significa "meglio": significa più violazioni *persistenti* rilevate; la composizione dice su
  quali categorie il detector fatica (tipicamente Glove/Shoe).

In [ ]:
tracker_results = {}
cap = cv.VideoCapture(VIDEO)
N_FRAMES = int(cap.get(cv.CAP_PROP_FRAME_COUNT))
cap.release()

for name in DETECTORS:
    print(f"\n=== {name} (skip={SKIP_FRAMES}) ===")
    tracker = build_tracker(
        detector=name,
        zones_file=ZONES_FILE,
        persistence_frames=PERSISTENCE,
        window_frames=WINDOW,
        skip_frames=SKIP_FRAMES,
        display=False,
        save_output=f"output/bench_{name}.mp4",
        alert_log=f"output/bench_{name}_alerts.json",
    )
    t0 = time.perf_counter()
    alerts = tracker.run(VIDEO)
    wall = time.perf_counter() - t0

    missing_counter = Counter()
    for a in alerts:
        for v in a.violations:
            missing_counter.update(v.missing_ppe)

    tracker_results[name] = {
        "wall_s": wall,
        "fps": N_FRAMES / wall,
        "alerts": len(alerts),
        "tracks": tracker.tracks_created,
        "missing": dict(missing_counter),
        "zone_violations": sum(len(a.zone_violations) for a in alerts),
    }
    print(f"-> {wall:.0f}s | {N_FRAMES / wall:.2f} fps effettivi | {len(alerts)} alert | {tracker.tracks_created} track")

In [ ]:
# ── Report sezione 1 ──────────────────────────────────────────────────────────
print(f"{'metrica':34} " + "".join(f"| {n:^18} " for n in DETECTORS))
print("-" * (35 + 21 * len(DETECTORS)))
for key, label, fmt in [
    ("wall_s", "tempo totale (s)", "{:18.0f}"),
    ("fps", "FPS effettivi", "{:18.2f}"),
    ("alerts", "alert confermati", "{:18d}"),
    ("tracks", "track creati (stabilita')", "{:18d}"),
    ("zone_violations", "violazioni zona", "{:18d}"),
]:
    print(f"{label:34} " + "".join("| " + fmt.format(tracker_results[n][key]) + " " for n in DETECTORS))
print(f"{'PPE mancanti negli alert':34} " + "".join(f"| {str(tracker_results[n]['missing']):^18.18} " for n in DETECTORS))
for n in DETECTORS:
    print(f"  dettaglio {n}: {tracker_results[n]['missing']}")

## 2 · Frame-level contro la weak ground truth

Campiona frame equispaziati e confronta le uscite del detector (+ associazione PPE↔persona del
checker) con i **fatti dichiarati** in `WEAK_GT`:

- **person_count_ok** — % di frame con un numero di persone nel range atteso.
- **helmet/vest coverage** — % di persone rilevate a cui il checker associa un casco/gilet.
  Se nel video *tutti* li indossano, ogni punto sotto il 100% è un falso negativo
  (di detection o di associazione spaziale).
- **flicker** — |Δ conteggio| medio tra frame campionati consecutivi: misura la stabilità temporale
  (meno = detection più stabili = meno lavoro per la sliding window).
- **latenza per frame**.

In [ ]:
cap = cv.VideoCapture(VIDEO)
frame_idxs = [int(i) for i in np.linspace(0, N_FRAMES - 1, VIDEO_SAMPLES)]
frames = []
for fi in frame_idxs:
    cap.set(cv.CAP_PROP_POS_FRAMES, fi)
    ret, frame = cap.read()
    if ret:
        frames.append((fi, frame))
fh_, fw_ = frames[0][1].shape[:2]
cap.release()
checker = PersonPPEChecker()

frame_results = {}
for name in DETECTORS:
    det = build_detector(name)
    rows = []
    print(f"\n=== {name} ===")
    for fi, frame in frames:
        res = det.analyze(frame)
        persons = checker.check(res.detections, fw_, fh_) if not res.error else []
        counts = Counter(d.label for d in res.detections)
        rows.append({
            "frame": fi,
            "ms": res.inference_time_ms,
            "n_persons": len(persons),
            "with_helmet": sum(1 for p in persons if p.found_ppe.get("Helmet", 0) >= 1),
            "with_vest": sum(1 for p in persons if p.found_ppe.get("Vest", 0) >= 1),
            "counts": dict(counts),
        })
        print(f"  frame {fi:>4}: {res.inference_time_ms:>7.0f}ms  persone={len(persons)} "
              f"casco={rows[-1]['with_helmet']} gilet={rows[-1]['with_vest']}")
    frame_results[name] = rows

In [ ]:
# ── Report sezione 2 ──────────────────────────────────────────────────────────
def flicker(rows, label):
    seq = [r["counts"].get(label, 0) for r in rows]
    return float(np.mean(np.abs(np.diff(seq)))) if len(seq) > 1 else 0.0


lo, hi = WEAK_GT["persons_range"]
print(f"weak GT: persone in [{lo},{hi}], casco per tutti={WEAK_GT['helmet_worn_by_all']}, "
      f"gilet per tutti={WEAK_GT['vest_worn_by_all']}\n")
print(f"{'metrica':34} " + "".join(f"| {n:^18} " for n in DETECTORS))
print("-" * (35 + 21 * len(DETECTORS)))

summary2 = {}
for name in DETECTORS:
    rows = frame_results[name]
    tot_persons = sum(r["n_persons"] for r in rows)
    summary2[name] = {
        "count_ok": np.mean([lo <= r["n_persons"] <= hi for r in rows]),
        "helmet_cov": (sum(r["with_helmet"] for r in rows) / tot_persons) if tot_persons else 0.0,
        "vest_cov": (sum(r["with_vest"] for r in rows) / tot_persons) if tot_persons else 0.0,
        "flicker_p": flicker(rows, "Person"),
        "lat": np.mean([r["ms"] for r in rows]),
    }

for key, label, fmt in [
    ("count_ok", "person count nel range (%)", "{:17.0%} "),
    ("helmet_cov", "helmet coverage (%)", "{:17.0%} "),
    ("vest_cov", "vest coverage (%)", "{:17.0%} "),
    ("flicker_p", "flicker Person (delta medio)", "{:17.2f} "),
    ("lat", "latenza media (ms/frame)", "{:17.0f} "),
]:
    print(f"{label:34} " + "".join("| " + fmt.format(summary2[n][key]) for n in DETECTORS))

# ── Timeline conteggio persone ────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 3.5))
for name in DETECTORS:
    ax.plot([r["frame"] for r in frame_results[name]],
            [r["n_persons"] for r in frame_results[name]], marker="o", label=name)
ax.axhspan(lo, hi, alpha=0.12, color="green", label=f"atteso [{lo},{hi}]")
ax.set_xlabel("frame"); ax.set_ylabel("persone rilevate"); ax.legend(); ax.grid(alpha=0.3)
ax.set_title("Conteggio persone per frame vs atteso (weak GT)")
plt.tight_layout(); plt.show()

## 3 · Fallback: ground truth vera su immagini etichettate

Il test split del dataset Roboflow (`data/test`, CC BY 4.0) ha bounding box annotate a mano:
qui le metriche sono le classiche **Precision / Recall / F1 per classe @ IoU 0.5** (matching greedy
per IoU decrescente). Meno fedele all'uso video, ma è l'unica ground truth "vera" disponibile.

**Attenzione — annotazioni parziali.** Questo dataset è un *merge* di più dataset con schemi
diversi: molte immagini annotano una sola classe (es. solo `Hardhat`) e solo 95/4423 annotano
`Person`. Valutare tutte le classi su ogni immagine produrrebbe falsi positivi fasulli (il detector
vede la persona, il GT non la annota). Per questo la valutazione è **stratificata e class-aware**:
per ogni classe si campionano `K_PER_CLASS` immagini che *contengono quella classe in GT*, e su
ogni immagine si valutano **solo le classi effettivamente annotate** in quell'immagine.

Le classi del dataset vengono mappate alle canoniche via keyword; le classi negative (`NO-*`) e
quelle fuori scopo (Mask, Goggles, Ladder, ...) sono ignorate. **Nota**: questo dataset non ha
scarpe → `Shoe` non è valutabile qui.

In [ ]:
if not (DATA_DIR / "data.yaml").exists():
    print("Dataset assente - scaricalo con: python -m src.visual_security.download_data")
    gt_results = None
else:
    with open(DATA_DIR / "data.yaml", encoding="utf-8") as f:
        DATASET_CLASSES = yaml.safe_load(f)["names"]

    def map_to_canonical(name: str) -> str | None:
        n = name.lower().replace("_", "-")
        if n.startswith(("no-", "non-")):
            return None
        for kw, label in [("person", "Person"), ("hardhat", "Helmet"), ("helmet", "Helmet"),
                          ("vest", "Vest"), ("glove", "Glove"), ("shoe", "Shoe"), ("boot", "Shoe")]:
            if kw in n:
                return label
        return None

    CLASS_MAP = {i: map_to_canonical(c) for i, c in enumerate(DATASET_CLASSES)}
    print("mapping:", {DATASET_CLASSES[i]: v for i, v in CLASS_MAP.items() if v})

    def gt_classes(img_path):
        """Classi canoniche annotate nella label di un'immagine."""
        lp = DATA_DIR / "test" / "labels" / (img_path.stem + ".txt")
        out = set()
        for line in (lp.read_text().splitlines() if lp.exists() else []):
            parts = line.split()
            if parts:
                label = CLASS_MAP.get(int(parts[0]))
                if label:
                    out.add(label)
        return out

    # ── Campionamento stratificato: K immagini per ogni classe canonica ──────
    test_images = sorted((DATA_DIR / "test" / "images").glob("*.jpg"))
    rng = np.random.default_rng(SEED)
    rng.shuffle(test_images)

    by_class = {}
    for img_path in test_images:
        for c in gt_classes(img_path):
            by_class.setdefault(c, []).append(img_path)

    sample_imgs = []
    for c, imgs in sorted(by_class.items()):
        picked = imgs[:K_PER_CLASS]
        sample_imgs.extend(p for p in picked if p not in sample_imgs)
        print(f"  classe {c}: {len(imgs)} immagini disponibili -> ne uso {len(picked)}")
    print(f"totale immagini da valutare: {len(sample_imgs)}")

    def load_gt(img_path):
        label_path = DATA_DIR / "test" / "labels" / (img_path.stem + ".txt")
        img = cv.imread(str(img_path)); h, w = img.shape[:2]
        out = []
        for line in (label_path.read_text().splitlines() if label_path.exists() else []):
            parts = line.split()
            if len(parts) < 5:
                continue
            label = CLASS_MAP.get(int(parts[0]))
            if label is None:
                continue
            cx, cy, bw, bh = (float(v) for v in parts[1:5])
            out.append((label, ((cx - bw / 2) * w, (cy - bh / 2) * h, (cx + bw / 2) * w, (cy + bh / 2) * h)))
        return out

    def iou(a, b):
        ix1, iy1 = max(a[0], b[0]), max(a[1], b[1])
        ix2, iy2 = min(a[2], b[2]), min(a[3], b[3])
        inter = max(0.0, ix2 - ix1) * max(0.0, iy2 - iy1)
        if inter == 0:
            return 0.0
        ua = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
        return inter / ua if ua > 0 else 0.0

    CANONICAL = ["Person", "Helmet", "Vest", "Glove", "Shoe"]

    def match_per_class(preds, gts):
        result = {}
        for label in CANONICAL:
            p = [b for l, b in preds if l == label]
            g = [b for l, b in gts if l == label]
            pairs = sorted(((iou(pb, gb), i, j) for i, pb in enumerate(p) for j, gb in enumerate(g)),
                           key=lambda t: t[0], reverse=True)
            used_p, used_g, tp = set(), set(), 0
            for s, i, j in pairs:
                if s < IOU_THR or i in used_p or j in used_g:
                    continue
                used_p.add(i); used_g.add(j); tp += 1
            result[label] = (tp, len(p) - tp, len(g) - tp)
        return result

    gt_results = {}
    for name in DETECTORS:
        print(f"\n=== {name} ===")
        det = build_detector(name)
        totals = {label: [0, 0, 0] for label in CANONICAL}
        lat = []
        for k, img_path in enumerate(sample_imgs, 1):
            res = det.analyze(str(img_path))
            if res.error:
                print(f"  [{k}] ERRORE: {res.error}")
                continue
            lat.append(res.inference_time_ms)
            preds = [(d.label, tuple(d.bbox)) for d in res.detections if d.bbox]
            annotated = gt_classes(img_path)  # valuta SOLO le classi annotate qui
            for label, (tp, fp, fn) in match_per_class(preds, load_gt(img_path)).items():
                if label not in annotated:
                    continue
                totals[label][0] += tp; totals[label][1] += fp; totals[label][2] += fn
            print(f"  [{k:>2}/{len(sample_imgs)}] {res.inference_time_ms:>7.0f}ms  {len(preds)} pred  (classi GT: {sorted(annotated)})")
        gt_results[name] = {"totals": totals, "lat": lat}

In [ ]:
if gt_results:
    def prf(tp, fp, fn):
        p = tp / (tp + fp) if tp + fp else 0.0
        r = tp / (tp + fn) if tp + fn else 0.0
        return p, r, (2 * p * r / (p + r) if p + r else 0.0)

    # solo classi con supporto nel campione
    visible = [c for c in ["Person", "Helmet", "Vest", "Glove", "Shoe"]
               if any(gt_results[n]["totals"][c][0] + gt_results[n]["totals"][c][2] > 0 for n in DETECTORS)]

    print(f"{'classe':10} " + "".join(f"| {n:^29} " for n in DETECTORS))
    print(f"{'':10} " + "| {:>7} {:>7} {:>7} {:>5} ".format("P", "R", "F1", "sup") * len(DETECTORS))
    print("-" * (11 + 32 * len(DETECTORS)))
    f1s = {n: [] for n in DETECTORS}
    for label in visible:
        row = f"{label:10} "
        for n in DETECTORS:
            tp, fp, fn = gt_results[n]["totals"][label]
            p, r, f = prf(tp, fp, fn)
            f1s[n].append(f)
            row += f"| {p:7.2f} {r:7.2f} {f:7.2f} {tp + fn:5d} "
        print(row)
    print("-" * (11 + 32 * len(DETECTORS)))
    row = f"{'micro':10} "
    for n in DETECTORS:
        tp = sum(gt_results[n]["totals"][c][0] for c in visible)
        fp = sum(gt_results[n]["totals"][c][1] for c in visible)
        fn = sum(gt_results[n]["totals"][c][2] for c in visible)
        p, r, f = prf(tp, fp, fn)
        row += f"| {p:7.2f} {r:7.2f} {f:7.2f} {tp + fn:5d} "
    print(row)
    for n in DETECTORS:
        print(f"latenza {n}: media {np.mean(gt_results[n]['lat']):.0f}ms")

    fig, ax = plt.subplots(figsize=(7, 3.5))
    x = np.arange(len(visible)); w = 0.8 / len(DETECTORS)
    for i, n in enumerate(DETECTORS):
        ax.bar(x + i * w, f1s[n], w, label=n)
    ax.set_xticks(x + w / 2, visible); ax.set_ylabel("F1 @ IoU 0.5"); ax.set_ylim(0, 1)
    ax.set_title("F1 per classe su ground truth etichettata"); ax.legend(); ax.grid(axis="y", alpha=0.3)
    plt.tight_layout(); plt.show()

## Come leggere i risultati

- **Sezione 1** è il verdetto operativo: a parità di parametri, guardare *FPS effettivi* (costo),
  *track creati* (stabilità identità: per 2-3 persone reali, valori vicini a 2-3 = ottimo, decine =
  identità che si perdono) e la *composizione degli alert* (se un detector genera alert solo su
  Glove/Shoe, sta faticando sugli oggetti piccoli, non sulle persone).
- **Sezione 2** traduce la weak GT in numeri: `helmet/vest coverage` sotto il 100% (con tutti che li
  indossano) = falsi negativi; `person count` fuori range = detection mancate o fantasma.
- **Sezione 3** è il confronto classico da laboratorio, utile per confrontare con la letteratura,
  ma meno rappresentativo dell'uso su video.
- Per un verdetto più solido: alzare `N_IMAGES`/`VIDEO_SAMPLES` (limite: ~22s/inferenza di
  Grounding DINO su CPU) o eseguire su GPU.